# Diabetes Prediction - Exploratory Data Analysis
## Experiment Tracking with MLflow on DagsHub

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import os
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Create artifacts directory
os.makedirs('../models/artifacts', exist_ok=True)

print("Libraries imported successfully!")

## 2. MLflow Configuration

In [ ]:
# Configure MLflow to use DagsHub
mlflow.set_tracking_uri('https://dagshub.com/taqihaider7/Diabetes-Prediction-DCS-NSU.mlflow')

# Set your DagsHub credentials as environment variables
# os.environ['MLFLOW_TRACKING_USERNAME'] = 'your_username'
# os.environ['MLFLOW_TRACKING_PASSWORD'] = 'your_token'

# Set experiment name
mlflow.set_experiment('diabetes-eda-experiments')

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('diabetes-eda-experiments')}")

## 3. Load Data

In [ ]:
# Load the dataset
df = pd.read_csv('../data/diabetes.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

## 4. EDA Experiment 1: Basic Statistical Analysis

In [ ]:
with mlflow.start_run(run_name='EDA_Basic_Statistics'):
    
    # Log parameters
    mlflow.log_param('dataset_name', 'diabetes.csv')
    mlflow.log_param('analysis_type', 'basic_statistics')
    
    # Basic info
    print("Dataset Info:")
    print(df.info())
    
    # Log metrics
    mlflow.log_metric('total_samples', len(df))
    mlflow.log_metric('total_features', len(df.columns))
    mlflow.log_metric('missing_values', df.isnull().sum().sum())
    
    # Class distribution
    outcome_dist = df['Outcome'].value_counts()
    mlflow.log_metric('class_0_count', int(outcome_dist[0]))
    mlflow.log_metric('class_1_count', int(outcome_dist[1]))
    mlflow.log_metric('class_imbalance_ratio', float(outcome_dist[0] / outcome_dist[1]))
    
    # Descriptive statistics
    desc_stats = df.describe()
    print("\nDescriptive Statistics:")
    print(desc_stats)
    
    # Save statistics
    desc_stats.to_csv('../models/artifacts/descriptive_stats.csv')
    mlflow.log_artifact('../models/artifacts/descriptive_stats.csv')
    
    # Class distribution plot
    plt.figure(figsize=(8, 6))
    sns.countplot(data=df, x='Outcome')
    plt.title('Class Distribution')
    plt.xlabel('Outcome (0: No Diabetes, 1: Diabetes)')
    plt.ylabel('Count')
    plt.savefig('../models/artifacts/class_distribution.png', dpi=300, bbox_inches='tight')
    mlflow.log_artifact('../models/artifacts/class_distribution.png')
    plt.show()
    
    print("\n✓ Basic statistics experiment logged to MLflow")

## 5. Data Quality Analysis

In [ ]:
# Check for zero values (biological impossibilities)
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("Zero values in critical columns:")
for col in zero_cols:
    zero_count = (df[col] == 0).sum()
    zero_pct = (zero_count / len(df)) * 100
    print(f"{col}: {zero_count} ({zero_pct:.2f}%)")

## 6. EDA Experiment 2: Feature Correlation Analysis

In [ ]:
with mlflow.start_run(run_name='EDA_Correlation_Analysis'):
    
    # Log parameters
    mlflow.log_param('dataset_name', 'diabetes.csv')
    mlflow.log_param('analysis_type', 'correlation_analysis')
    
    # Correlation matrix
    corr_matrix = df.corr()
    
    # Log correlation with target
    target_corr = corr_matrix['Outcome'].sort_values(ascending=False)
    print("Correlation with Outcome:")
    print(target_corr)
    
    for feature, corr_value in target_corr.items():
        if feature != 'Outcome':
            mlflow.log_metric(f'corr_{feature}_outcome', float(corr_value))
    
    # Correlation heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.savefig('../models/artifacts/correlation_matrix.png', dpi=300, bbox_inches='tight')
    mlflow.log_artifact('../models/artifacts/correlation_matrix.png')
    plt.show()
    
    # Feature distributions by outcome
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.ravel()
    
    for idx, col in enumerate(df.columns[:-1]):
        axes[idx].hist([df[df['Outcome']==0][col], df[df['Outcome']==1][col]], 
                      label=['No Diabetes', 'Diabetes'], bins=20, alpha=0.7)
        axes[idx].set_title(col)
        axes[idx].legend()
    
    plt.tight_layout()
    plt.savefig('../models/artifacts/feature_distributions.png', dpi=300, bbox_inches='tight')
    mlflow.log_artifact('../models/artifacts/feature_distributions.png')
    plt.show()
    
    # Box plots for outlier detection
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.ravel()
    
    for idx, col in enumerate(df.columns[:-1]):
        axes[idx].boxplot([df[df['Outcome']==0][col], df[df['Outcome']==1][col]], 
                         labels=['No Diabetes', 'Diabetes'])
        axes[idx].set_title(col)
    
    plt.tight_layout()
    plt.savefig('../models/artifacts/outlier_boxplots.png', dpi=300, bbox_inches='tight')
    mlflow.log_artifact('../models/artifacts/outlier_boxplots.png')
    plt.show()
    
    print("\n✓ Correlation analysis experiment logged to MLflow")

## 7. Summary Statistics by Outcome

In [ ]:
# Group statistics by outcome
print("Mean values by Outcome:")
outcome_stats = df.groupby('Outcome').mean()
print(outcome_stats)

# Save to artifacts
outcome_stats.to_csv('../models/artifacts/outcome_statistics.csv')

## 8. Key Insights

In [ ]:
insights = f"""
=== KEY INSIGHTS FROM EDA ===

1. Dataset Overview:
   - Total samples: {len(df)}
   - Features: {len(df.columns) - 1}
   - Class 0 (No Diabetes): {outcome_dist[0]} ({outcome_dist[0]/len(df)*100:.1f}%)
   - Class 1 (Diabetes): {outcome_dist[1]} ({outcome_dist[1]/len(df)*100:.1f}%)
   - Class imbalance ratio: {outcome_dist[0]/outcome_dist[1]:.2f}:1

2. Top Features Correlated with Diabetes:
{target_corr.head(4).to_string()}

3. Data Quality:
   - Zero values found in biological features (need handling)
   - No missing values in dataset

4. Recommendations:
   - Handle zero values in Glucose, BloodPressure, BMI, etc.
   - Consider feature scaling (different ranges)
   - Address class imbalance in modeling
   - Focus on top correlated features: Glucose, BMI, Age
"""

print(insights)

# Save insights
with open('../models/artifacts/eda_insights.txt', 'w') as f:
    f.write(insights)

print("\n✓ EDA complete! All artifacts saved to models/artifacts/")